# Activity: Fair Today, Fair Tomorrow? — FreshKart PayLater Audit

**Unit 4 · AI-Assisted Visualization · Bias, Model Drift and Responsible AI Visualization**
Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## The case

FreshKart launched **PayLater**: customers can buy groceries now and pay at the end of the month. An AI model scores every application; applications with **score ≥ 0.5 are approved** for a full limit.

During 2026 FreshKart ran a **pilot**: *every* applicant received at least a small limit, so we know who repaid, whether the model approved them or not. Repayment is due two months later, so **November and December outcomes are not known yet**.

* The model was trained on **2025** applications: Urban 62%, Semi-urban 30%, **Rural 8%**.
* In **July 2026** marketing launched a **cashback promo** ("order more, earn more").

Risk head **Farah** received the monthly summary below from **AutoInsights**, an AI analytics assistant, and must present it to the board. **You are the auditor.** Check each statement, then design what Farah should show instead.

| # | AutoInsights statement |
|---|---|
| 1 | "PayLater approved **62%** of applicants in 2026." |
| 2 | "**Rural applicants repay less**, so the model's lower approval rate for them is justified and fair." |
| 3 | "Approved customers repay as reliably as ever: **about 70% of approved customers repay**, stable all year." |
| 4 | "The July cashback promo made customers **more creditworthy**: average scores rose after July." |
| 5 | "**November and December had zero defaults** among approved customers — the best months of the year." |
| 6 | "Deep dive: **Urban applicants aged 53** who applied in February repay **93%** of the time, far above average (p = 0.01). Target them!" |
| 7 | "No retraining is needed: the inputs look the same as in training." |

**Ground rules (from the lecture):** compare groups with counts and intervals · beware aggregation (Simpson's paradox) · drift: inputs *and* outcomes, with early proxies because labels arrive late · verify every generated number and causal claim · a figure must not mislead.

**How to work:** run the notebook top to bottom. Complete every cell marked `# TODO` (replace `None` with your code or answer) and write in the cells marked **✏️ Your answer**. Cells that depend on your TODO will tell you if something is still missing.

### The data: `freshkart_paylater.csv` (synthetic)

| Column | Meaning |
|---|---|
| `app_id`, `month` | application and the month it was made (2026-01 … 2026-12) |
| `region` | Urban, Semi-urban or Rural delivery address |
| `age`, `monthly_income_k` | applicant age; declared monthly income (₹ thousand) |
| `orders_90d`, `tenure_months` | FreshKart orders in the last 90 days; months as a customer |
| `score`, `approved` | model score (0–1) and decision (1 if score ≥ 0.5) |
| `repaid` | 1 = repaid on time, 0 = did not; **blank if not yet due** |

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (6, 3.2), "axes.spines.top": False, "axes.spines.right": False})
GREY, ACCENT, BLUE = "#bdbdbd", "#d95f02", "#4c78a8"
RCOL = {"Urban": BLUE, "Semi-urban": GREY, "Rural": ACCENT}
REGIONS = ["Urban", "Semi-urban", "Rural"]

def ready(*vals):
    """Helper: tells you if a TODO above is still unanswered."""
    ok = all(v is not None for v in vals)
    if not ok:
        print("⚠ Complete the TODO cell(s) above first, then re-run this cell.")
    return ok

df = pd.read_csv("freshkart_paylater.csv")
labelled = df.dropna(subset=["repaid"]).copy()
h1 = labelled[labelled.month <= "2026-06"].copy()          # Jan-Jun: before the promo, outcomes known
df.head()

---
## Part 1 · Who is in the data, and who is missing? (statements 1 and 5)

In [ ]:
approval_rate = None   # TODO: share of ALL 2026 applications that were approved
label_share = None     # TODO: share of applications whose repayment outcome is known

if ready(approval_rate, label_share):
    print(f"approved: {approval_rate:.1%} | outcome known: {label_share:.1%}")
    comp = pd.DataFrame({"2025 training data": pd.Series({"Urban": .62, "Semi-urban": .30, "Rural": .08}),
                         "2026 applicants": df.region.value_counts(normalize=True)}).loc[REGIONS]
    fig, ax = plt.subplots(1, 2, figsize=(13, 3))
    comp.T.plot.barh(stacked=True, ax=ax[0], color=[RCOL[r] for r in REGIONS]); ax[0].set_xlim(0, 1); ax[0].legend(ncol=3, fontsize=8, loc="lower right")
    ax[0].set_title("Region mix: training data vs today's applicants", loc="left")
    known = df.groupby("month").repaid.apply(lambda s: s.notna().mean())
    ax[1].bar(known.index, known.values, color=[GREY if v == 1 else ACCENT for v in known.values]); ax[1].tick_params(axis="x", rotation=60)
    ax[1].set_title("Share of applications with a known outcome", loc="left"); plt.tight_layout(); plt.show()

**✏️ Your answer (Part 1):** (a) Which kind of bias does the region comparison reveal (data, model, visualization or interpretation)? (b) How could AutoInsights have produced statement 5, and what should a chart of Nov–Dec repayment show instead?

*Your answer:*

---
## Part 2 · Selection bias: "just retrain on the people we approved"

After the pilot, FreshKart will only learn outcomes for **approved** customers. The data team proposes retraining on those alone. What happens to the relationship between **income and repayment** if we only see approved applicants?

In [ ]:
x_all, y_all = np.log(h1.monthly_income_k), h1.repaid
appr = h1.approved == 1

r_all = None        # TODO: correlation between log income and repaid, all Jan-Jun applicants (hint: np.corrcoef(x_all, y_all)[0, 1])
r_approved = None   # TODO: the same, approved applicants only

if ready(r_all, r_approved):
    print(f"r(all) = {r_all:.3f} | r(approved only) = {r_approved:.3f}")
    fig, ax = plt.subplots(figsize=(7, 3.2))
    bins = pd.qcut(h1.monthly_income_k, 10)
    for lab, m, col in [("all applicants", np.ones(len(h1), bool), GREY), ("approved only", appr.values, BLUE)]:
        t = h1[m].groupby(bins[m], observed=True).agg(inc=("monthly_income_k", "median"), rep=("repaid", "mean"))
        ax.plot(t.inc, t.rep, marker="o", color=col, label=lab)
    ax.set_xscale("log"); ax.set_xlabel("monthly income (₹k, log)"); ax.set_ylabel("repaid share"); ax.legend(); plt.show()

**✏️ Your answer (Part 2):** Why does the relationship look weaker among approved customers, and what would a model retrained only on them learn?

*Your answer:*

---
## Part 3 · Fairness by region (statement 2)

Use **Jan–Jun** (before the promo). For each region compute the measures from the lecture. "Positive" = repaid.

In [ ]:
def group_table(d):
    out = {}
    for r in REGIONS:
        t = d[d.region == r]
        y, p = t.repaid == 1, t.approved == 1
        out[r] = {
            "n": len(t),
            "repaid rate (base rate)": y.mean(),
            "selection rate": None,   # TODO: share approved
            "TPR": None,              # TODO: approved AND repaid / repaid
            "FPR": None,              # TODO: approved AND not repaid / not repaid
            "precision": None,        # TODO: approved AND repaid / approved
        }
    return pd.DataFrame(out).T

gt = group_table(h1)
if ready(*gt.drop(columns="n").values.ravel()):
    display(gt.astype(float).round(3))
    fig, ax = plt.subplots(1, 4, figsize=(17, 3), sharey=True)
    for a, m in zip(ax, ["selection rate", "TPR", "FPR", "precision"]):
        a.barh(REGIONS, gt[m].astype(float), color=[RCOL[r] for r in REGIONS])
        for i, r in enumerate(REGIONS): a.text(float(gt.loc[r, m]) + .01, i, f"{float(gt.loc[r, m]):.2f}", va="center", fontsize=9)
        a.set_xlim(0, 1.1); a.set_title(m)
    plt.tight_layout(); plt.show()

selection_ratio = None    # TODO: Rural selection rate / Urban selection rate
eo_gap = None             # TODO: equal-opportunity gap = Urban TPR - Rural TPR

if ready(selection_ratio, eo_gap):
    print(f"selection ratio Rural/Urban = {selection_ratio:.2f} (four-fifths rule of thumb: 0.8) | equal-opportunity gap = {eo_gap:.2f}")

In [ ]:
# Group calibration: does a score mean the same thing in every region?
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot([0, 1], [0, 1], "k--", lw=.8)
for r in REGIONS:
    t = h1[h1.region == r]
    b = t.groupby(pd.cut(t.score, np.linspace(0, 1, 9)), observed=True).agg(s=("score", "mean"), y=("repaid", "mean"), n=("repaid", "size"))
    b = b[b.n >= 25]
    ax.errorbar(b.s, b.y, yerr=1.96 * np.sqrt(b.y * (1 - b.y) / b.n), fmt="o-", color=RCOL[r], capsize=3, label=r)
ax.set_xlabel("model score"); ax.set_ylabel("actual repaid share"); ax.legend(); plt.show()

**✏️ Your answer (Part 3):** Which fairness measure(s) does the model fail for Rural applicants? Use the calibration plot to explain *how* the model treats a Rural applicant with the same score as an Urban one.

*Your answer:*

---
## Part 4 · "Rural applicants repay less": Simpson's paradox?

In [ ]:
h1["income_band"] = pd.cut(h1.monthly_income_k, [0, 25, 40, 1000], labels=["< ₹25k", "₹25–40k", "> ₹40k"])

pooled = None     # TODO: repaid rate by region, all incomes together (a Series indexed by region)
by_band = None    # TODO: repaid rate for every (income_band, region) pair as a table: rows = income band, columns = region (hint: pivot_table)
band_mix = pd.crosstab(h1.region, h1.income_band, normalize="index").round(2)

if ready(pooled, by_band):
    display(pooled.round(3).rename("pooled repaid rate")); display(by_band.round(3)); display(band_mix)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
    ax[0].bar(REGIONS, pooled.loc[REGIONS], color=[RCOL[r] for r in REGIONS]); ax[0].set_title("Pooled"); ax[0].set_ylim(0.4, 1)
    by_band[REGIONS].plot.bar(ax=ax[1], color=[RCOL[r] for r in REGIONS], rot=0); ax[1].set_title("Within each income band")
    plt.tight_layout(); plt.show()

rural_better_in_every_band = None   # TODO: True/False

**✏️ Your answer (Part 4):** Rewrite statement 2 so it is honest. What is the lurking variable, and why does the pooled chart mislead?

*Your answer:*

---
## Part 5 · Could a different threshold close the gap?

In [ ]:
grid = np.round(np.arange(0.02, 0.81, 0.02), 2)
rural, urban = h1[h1.region == "Rural"], h1[h1.region == "Urban"]
def rates(t, thr):
    p, y = t.score >= thr, t.repaid == 1
    return p.mean(), (p & y).sum() / y.sum(), (p & ~y).sum() / (~y).sum()

urban_tpr = rates(urban, 0.5)[1]
rural_tprs = np.array([rates(rural, g)[1] for g in grid])

rural_threshold = None    # TODO: the threshold in grid whose Rural TPR is closest to urban_tpr

if ready(rural_threshold):
    t = pd.DataFrame({"Urban @ 0.50": rates(urban, .5), "Rural @ 0.50": rates(rural, .5), f"Rural @ {rural_threshold}": rates(rural, rural_threshold)},
                     index=["selection rate", "TPR", "FPR"]).round(3)
    display(t)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3))
    for k, lab in enumerate(["selection rate", "TPR", "FPR"]):
        for nm, d in [("Urban", urban), ("Rural", rural)]:
            ax[k].plot(grid, [rates(d, g)[k] for g in grid], color=RCOL[nm], label=nm)
        ax[k].axvline(.5, color="k", ls="--", lw=.8); ax[k].axvline(rural_threshold, color=ACCENT, ls=":"); ax[k].set_title(lab); ax[k].set_xlabel("threshold")
    ax[0].legend(); plt.tight_layout(); plt.show()

**✏️ Your answer (Part 5):** What does the lower Rural threshold fix, and what does it cost? Who at FreshKart should decide, and what must the chart show them?

*Your answer:*

---
## Part 6 · Drift after the July promo (statements 3, 4 and 7)

### 6a · Data drift: PSI of each input, month by month (reference = Jan–Mar)

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1))); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(ref, edges)[0] / len(ref)
    a = np.histogram(cur, edges)[0] / len(cur)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return None   # TODO: the PSI formula, sum of (a - e) * ln(a / e)

ref = df[df.month <= "2026-03"]
features = ["monthly_income_k", "orders_90d", "tenure_months", "age"]
months = sorted(df.month.unique())
if psi(ref.age, ref.age) is not None:
    drift = pd.DataFrame({f: [psi(ref[f], df[df.month == m][f]) for m in months] for f in features}, index=months).round(3)
    display(drift.style.background_gradient(cmap="Oranges", vmin=0, vmax=.4).format("{:.2f}"))

drifted_feature = None   # TODO: which feature drifts past the 0.25 alarm level?
first_alarm_month = None # TODO: the first month its PSI exceeds 0.25 (e.g. '2026-07')

### 6b · Early proxy vs delayed truth

In [ ]:
monthly = df.groupby("month").agg(n=("app_id", "size"), mean_score=("score", "mean"), approval=("approved", "mean"))
acc = labelled[labelled.approved == 1].groupby("month").repaid.agg(["mean", "size"])   # repaid share among APPROVED customers

monthly["approved_repaid"] = acc["mean"]
monthly["acc_se"] = None    # TODO: standard error of each month's approved-repaid share: sqrt(p(1-p)/n) using acc["mean"] and acc["size"]

if monthly["acc_se"].notna().any():
    base = monthly.loc["2026-01":"2026-06", "approved_repaid"].mean()
    fig, ax = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
    ax[0].plot(monthly.index, monthly.mean_score, marker="o", color=BLUE); ax[0].set_ylabel("mean score")
    ax[0].set_title("Available immediately: mean model score", loc="left")
    m = monthly.approved_repaid.notna()
    ax[1].errorbar(monthly.index[m], monthly.approved_repaid[m], yerr=1.96 * monthly.acc_se[m].astype(float), fmt="o-", color=ACCENT, capsize=3)
    ax[1].axhline(base, color="k", ls="--", lw=.8); ax[1].text(0, base + .005, f"Jan–Jun baseline {base:.2f}", fontsize=8)
    ax[1].set_title("Needs repayment outcomes (2-month delay): repaid share among approved, 95% intervals", loc="left"); ax[1].set_ylabel("approved who repaid")
    for a in ax: a.axvline("2026-07", color="#666", lw=1)
    ax[0].text("2026-07", ax[0].get_ylim()[1] * .98, " July promo", fontsize=8, va="top")
    plt.tight_layout(); plt.show()
    display(monthly.round(3))

In [ ]:
# Has the meaning of 'orders_90d' changed? Repaid share by orders band, before vs after the promo
lab = labelled.assign(period=np.where(labelled.month < "2026-07", "Jan–Jun", "Jul–Oct"),
                      orders_band=pd.cut(labelled.orders_90d, [0, 6, 12, 18, 25, 200], labels=["0–6", "7–12", "13–18", "19–25", "26+"]))
rel = lab.pivot_table(index="orders_band", columns="period", values="repaid", aggfunc="mean", observed=True)
display(rel.round(3))
rel.plot(marker="o", color=[GREY, ACCENT], title="Repaid share by orders in last 90 days"); plt.ylabel("repaid share"); plt.show()

drift_kinds = None   # TODO: a list with the kind(s) of drift you see after July: 'data drift', 'concept drift', 'label drift'

**✏️ Your answer (Part 6):** (a) Rewrite statements 3, 4 and 7. (b) Which signal would have warned Farah **first**, and which one **confirmed** the problem? (c) Write a takeaway **title** and a **caption** (window, statistic, threshold, data source) for the repaid-share chart, following Wilke Ch. 22.

*Your answer:*

---
## Part 7 · Is October really worse? (numerical)

In [ ]:
oct_ = labelled[(labelled.month == "2026-10") & (labelled.approved == 1)]
p_oct, n_oct = oct_.repaid.mean(), len(oct_)
baseline = h1[h1.approved == 1].repaid.mean()

se_oct = None        # TODO: standard error of October's accuracy
ci_low, ci_high = None, None   # TODO: 95% interval (± 1.96 SE)

if ready(se_oct, ci_low, ci_high):
    print(f"October: {p_oct:.3f} of approved repaid (n = {n_oct}), 95% interval {ci_low:.3f} to {ci_high:.3f}; Jan–Jun baseline {baseline:.3f}")
    print("baseline outside the interval?", not (ci_low <= baseline <= ci_high))

---
## Part 8 · Verify the AI's "deep dive" (statement 6)

AutoInsights scanned every combination of **region (3) × age (39 single years) × month (10 labelled months)** and reported the best one.

In [ ]:
segments = labelled.groupby(["region", "age", "month"]).repaid.agg(["mean", "size"])
segments = segments[segments["size"] >= 5]
overall = labelled.repaid.mean()
segments["p"] = [stats.binomtest(int(round(m * n)), int(n), overall).pvalue for m, n in zip(segments["mean"], segments["size"])]

n_tests = None            # TODO: number of segments tested (rows in `segments`)
expected_false = None     # TODO: expected 'significant' segments at alpha = 0.05 if nothing were real
bonferroni = None         # TODO: Bonferroni threshold

if ready(n_tests, expected_false, bonferroni):
    print(f"segments tested: {n_tests} | expected false findings: {expected_false:.1f} | Bonferroni threshold: {bonferroni:.6f}")
    print(f"segments with p < 0.05: {(segments.p < .05).sum()} | surviving Bonferroni: {(segments.p < bonferroni).sum()}")
    display(segments.sort_values("p").head(5).round(4))

### 8b · Check every statement against the data

In [ ]:
checks = pd.DataFrame({
    "statement": [1, 2, 3, 4, 5, 6, 7],
    "recomputed evidence": [
        f"approval rate = {df.approved.mean():.1%}",
        f"pooled repaid: {h1.groupby('region').repaid.mean().round(2).to_dict()} | see Part 4 by band",
        f"repaid share among approved, by month: {labelled[labelled.approved == 1].groupby('month').repaid.mean().round(2).to_dict()}",
        f"mean score Jan–Jun {df[df.month < '2026-07'].score.mean():.2f} vs Jul–Dec {df[df.month >= '2026-07'].score.mean():.2f}; repaid share Jan–Jun {labelled[labelled.month < '2026-07'].repaid.mean():.2f} vs Jul–Oct {labelled[labelled.month >= '2026-07'].repaid.mean():.2f}",
        f"Nov–Dec approved rows with a known outcome: {df[(df.month >= '2026-11') & (df.approved == 1)].repaid.notna().sum()}",
        "see Part 8a",
        "see Part 6a",
    ]})
pd.set_option("display.max_colwidth", 200); display(checks)

decisions = {
    1: None,   # 62% approved
    2: None,   # Rural repay less, so fair
    3: None,   # ~70% of approved repay, stable
    4: None,   # promo made customers more creditworthy
    5: None,   # zero defaults in Nov-Dec
    6: None,   # Urban, age 53, February segment
    7: None,   # no retraining needed
}   # TODO: 'Accept', 'Correct' or 'Reject'

if all(v in ("Accept", "Correct", "Reject") for v in decisions.values()):
    print(pd.Series(decisions).value_counts().to_string())
else:
    print("⚠ Fill in all seven decisions with 'Accept', 'Correct' or 'Reject'.")

---
## Part 9 · Responsible reporting: the board slide and the model card

### 9a · Focus attention honestly
The draft board chart below headlines overall accuracy. Redraw it so the **important** finding, not the **favourable** one, is what the eye sees first (Knaflic Ch. 4–5; Ware G1.2): grey the context, one accent colour, direct labels, a takeaway title.

In [ ]:
by_region_acc = h1.assign(c=h1.approved == h1.repaid).groupby("region").c.mean().loc[REGIONS]
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
ax[0].bar(["Overall"], [(h1.approved == h1.repaid).mean()], color="green", width=.4); ax[0].set_ylim(0, 1); ax[0].set_title("Draft: 'PayLater model is accurate'")

# TODO (drawing): on ax[1], show TPR by region (from Part 3) with Rural in ACCENT and the others in GREY, label each bar
# with its value, hide the y-axis, and give it a takeaway title that states the finding.

plt.tight_layout(); plt.show()

### 9b · Horizontal logic: the story in five headlines
Knaflic Ch. 7: write the headlines first; reading only them, in order, should tell the whole story, **including the unfavourable parts**.

In [ ]:
headlines = [
    None,   # TODO: 1 context
    None,   # TODO: 2 fairness finding
    None,   # TODO: 3 why (data / proxy)
    None,   # TODO: 4 drift finding
    None,   # TODO: 5 call to action
]
if ready(*headlines):
    print("\n".join(f"{i + 1}. {h}" for i, h in enumerate(headlines)))

### 9c · A model card (Mitchell et al. 2019)

In [ ]:
model_card = {
    "intended use": None,               # TODO
    "not intended for": None,           # TODO
    "training data": None,              # TODO: include the region mix
    "evaluation": None,                 # TODO: which months, which label
    "performance by group": None,       # TODO: numbers from Part 3
    "known limitations": None,          # TODO: at least two (fairness, drift, label delay...)
    "monitoring": None,                 # TODO: what is tracked, thresholds, who owns it
}
if ready(*model_card.values()):
    for k, v in model_card.items(): print(f"{k.upper():22s}: {v}")

**✏️ Your answer (Part 9):** Map your recommendations to the responsible-AI principles from the lecture (transparency, fairness, human oversight, robustness, accountability, privacy): one visualization practice for each.

*Your answer:*

---
## Reflection

Answer in 2–4 sentences each. There are no right-or-wrong answers here; we are looking for honest thinking about *your own* process.

**1. Which statement did you believe?** Before checking, which AutoInsights statement sounded most convincing? Was it the confident wording, the number, or the fact that it matched what you expected (confirmation bias)?

**2. Whose fairness?** Equal selection rates, equal TPR and equal precision cannot all hold here. Which one would you prioritise for PayLater, and who should have a say in that choice?

**3. Validation levels.** Using Munzner's nested model, name one threat at each level (wrong problem, wrong abstraction, wrong idiom, wrong algorithm) that you found or suspect in this case.

**4. After the audit.** The pilot ends in December: from January, outcomes will only be known for approved customers. How will that change what your monitoring dashboard can and cannot show, and what would you put in place now?

*Your reflection:*